# 04. BC を動かす（Command Job）

**対応するテキスト**: [docs/05_BCを動かす.md](../docs/05_BCを動かす.md)

**前提**: [03_collect_demos_job.ipynb](03_collect_demos_job.ipynb) が完了し、
データ資産 `il-pickplace-demos`（**800 エピソード以上**）が登録済みであること。

このノートブックでは、**勾配更新回数をそろえて、デモ数だけを変える**実験を行います。

> ⚠ **エポック数を固定してデモ数だけを変えると、学習量も一緒に変わってしまいます**（交絡）。
> 理由は [docs/05 の 5.3](../docs/05_BCを動かす.md) を先に読んでください。

> [!WARNING]
> **このノートブックの Azure ジョブは Azure 上で実行検証していません。**
> 本ハンズオンの構築時に検証したのは**ローカル実行だけ**です。
> Azure ジョブの所要時間・費用は記載していません。**あなたの環境で確認してください。**

> ⚠ **サブスクリプション ID を書き込んだノートブックをコミットしないでください。**

In [ ]:
# ============================================================
#  ここを自分の環境に書き換えてください（01・03 と同じ値）
# ============================================================
SUBSCRIPTION_ID = "<SUBSCRIPTION_ID>"
RESOURCE_GROUP = "<RESOURCE_GROUP>"
WORKSPACE_NAME = "<AML_WORKSPACE_NAME>"

COMPUTE_NAME = "cpu-cluster"
ENV_REF = "il-pickplace-env@latest"
DATA_ASSET_NAME = "il-pickplace-demos"
EXPERIMENT = "il-bc"

BATCH_SIZE = 32          # 本ノートブックでは固定（Sweep で振るのは 06 で行います）

TAGS = {
    "project": "il-workshop",
    "owner": "<your-alias>",
    "delete-after": "<YYYY-MM-DD>",
}

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
import mlflow

ml_client = MLClient(
    credential=DefaultAzureCredential(exclude_interactive_browser_credential=False),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)
ws = ml_client.workspaces.get(WORKSPACE_NAME)
print("接続しました:", ws.name)

tracking_uri = getattr(ws, "mlflow_tracking_uri", None)
if tracking_uri is None:
    tracking_uri = (
        f"azureml://{ws.location}.api.azureml.ms/mlflow/v1.0"
        f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
        f"/providers/Microsoft.MachineLearningServices/workspaces/{WORKSPACE_NAME}"
    )
mlflow.set_tracking_uri(tracking_uri)
print("MLflow 追跡先を設定しました。")

## 1. 実験条件を決める

$$
\text{勾配更新の回数} \;\approx\; \frac{\text{デモの総ステップ数}}{\text{バッチ サイズ}} \times \text{エポック数}
$$

1 エピソードは必ず 50 ステップなので、**デモ数からステップ数が決まります**。
そこから **更新回数が約 46,000 回になるエポック数**を逆算します。

In [ ]:
STEPS_PER_EPISODE = 50          # 固定ホライズン
CONDITIONS = [(50, 600), (200, 150), (800, 37)]
SEEDS = (0, 1, 2)

print(f"{'デモ数':>6}  {'ステップ数':>10}  {'エポック':>8}  {'更新回数':>10}")
for n_demos, epochs in CONDITIONS:
    transitions = n_demos * STEPS_PER_EPISODE
    print(f"{n_demos:6d}  {transitions:10d}  {epochs:8d}  {transitions // BATCH_SIZE * epochs:10d}")

print()
print("→ 3 条件とも更新回数はほぼ同じ。変えているのはデモ数だけです。")
print(f"→ 投入するジョブ数: {len(CONDITIONS) * len(SEEDS)} 本")

## 2. ジョブ投入をひとまとめにする

[../src/train_il.py](../src/train_il.py) に渡す引数のうち、
**実験で振るのは `--n-demo-episodes` / `--epochs` / `--seed`** です。

In [ ]:
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes

TRAIN_COMMAND = (
    "python train_il.py"
    " --algo bc"
    " --demos-dir ${{inputs.demos}}"
    " --output-dir ${{outputs.model}}"
    " --n-demo-episodes ${{inputs.n_demo_episodes}}"
    " --epochs ${{inputs.epochs}}"
    " --batch-size ${{inputs.batch_size}}"
    " --seed ${{inputs.seed}}"
)


def submit_bc(n_demo_episodes, epochs, seed):
    job = command(
        code="../src",
        command=TRAIN_COMMAND,
        inputs=dict(
            demos=Input(type=AssetTypes.URI_FOLDER, path=f"azureml:{DATA_ASSET_NAME}@latest"),
            n_demo_episodes=n_demo_episodes,
            epochs=epochs,
            batch_size=BATCH_SIZE,
            seed=seed,
        ),
        outputs=dict(model=Output(type=AssetTypes.URI_FOLDER)),
        environment=ENV_REF,
        compute=COMPUTE_NAME,
        experiment_name=EXPERIMENT,
        display_name=f"bc_demo{n_demo_episodes}_ep{epochs}_seed{seed}",
        tags={**TAGS, "algo": "bc"},
    )
    returned = ml_client.jobs.create_or_update(job)
    print(f"  投入: {returned.display_name}  ({returned.name})")
    return returned

import time


def wait_all(jobs, poll_seconds=30):
    # 投入した全ジョブが終了状態になるまで待つ
    pending = {job.name for job in jobs}
    while pending:
        finished = set()
        for name in sorted(pending):
            status = str(ml_client.jobs.get(name).status)
            if status in ("Completed", "Failed", "Canceled"):
                print(f"  {name}: {status}")
                finished.add(name)
        pending -= finished
        if pending:
            print(f"  ...残り {len(pending)} 本。{poll_seconds} 秒後に再確認します。")
            time.sleep(poll_seconds)
    print("すべて終了しました。")

## 3. 投入する（3 条件 × 3 シード = 9 本）

> ⚠ **1 つのシードの結果だけで比較してはいけません。**
> この題材では、**同じ設定でもシードを変えると成功率が 0.2 前後動きます**（[docs/05 の 5.6](../docs/05_BCを動かす.md)）。

> ⚠ **`--seed` は `src/il_common.py` の `set_seed()` で `random` / `numpy` / `torch` をまとめて固定します。**
> **これを行わないと、同じ seed でも結果が再現しません**（[docs/A1 の 3-2](../docs/A1_トラブルシューティング.md)）。

In [ ]:
print("BC ジョブを投入します:")
jobs = [submit_bc(n, epochs=e, seed=s) for n, e in CONDITIONS for s in SEEDS]

In [ ]:
wait_all(jobs)

## 4. 結果を集計する

同じ実験名に記録された Run は `mlflow.search_runs()` でまとめて取得できます。

> 出典（Microsoft 公式）: [Query & compare experiments and runs with MLflow](https://learn.microsoft.com/azure/machine-learning/how-to-track-experiments-mlflow?view=azureml-api-2)

In [ ]:
import pandas as pd

runs = mlflow.search_runs(experiment_names=[EXPERIMENT], output_format="pandas")
print("取得した Run 数:", len(runs))

COLS = {
    "tags.mlflow.runName": "run_name",
    "params.n_demo_episodes": "n_demos",
    "params.epochs": "epochs",
    "params.batch_size": "batch_size",
    "params.seed": "seed",
    "metrics.n_transitions": "n_transitions",
    "metrics.eval_success_rate": "success_rate",
    "metrics.eval_return_mean": "return_mean",
    "metrics.eval_return_std": "return_std",
    "metrics.normalized_return": "normalized",
    "metrics.expert_success_rate": "expert_success_rate",
}
available = {k: v for k, v in COLS.items() if k in runs.columns}
df = runs[list(available)].rename(columns=available).dropna(subset=["success_rate"])
for col in ("n_demos", "epochs", "batch_size", "seed"):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

sort_cols = [c for c in ("n_demos", "seed") if c in df.columns]
if sort_cols:
    df = df.sort_values(sort_cols)
df.to_csv("bc_results.csv", index=False, encoding="utf-8-sig")
df

In [ ]:
summary = (
    df.groupby("n_demos")[["success_rate", "return_mean", "normalized"]]
    .agg(["count", "mean", "min", "max"])
)
print("=== デモ数ごとの成績（更新回数はそろえてある） ===")
print(summary)

print()
print("=== 判定 ===")
spread = df.groupby("n_demos")["success_rate"].agg(lambda s: s.max() - s.min())
means = df.groupby("n_demos")["success_rate"].mean()
print("同一条件のばらつき（最大 - 最小）:")
print(spread)
print()
print(f"条件間の平均の最大差 : {means.max() - means.min():.3f}")
print(f"ばらつきの最大値     : {spread.max():.3f}")
if means.max() - means.min() > spread.max():
    print("→ 条件間の差は、同一条件のばらつきより大きい（差があると言える）")
else:
    print("→ 条件間の差は、ばらつきに埋もれている（差があるとは言えない）")

In [ ]:
import matplotlib.pyplot as plt

if len(df):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    agg = df.groupby("n_demos")["success_rate"].agg(["mean", "min", "max"]).sort_index()
    axes[0].errorbar(
        agg.index, agg["mean"],
        yerr=[agg["mean"] - agg["min"], agg["max"] - agg["mean"]],
        marker="o", capsize=4,
    )
    axes[0].set_xscale("log")
    axes[0].set_xlabel("デモの本数（対数軸）")
    axes[0].set_ylabel("eval_success_rate")
    axes[0].set_title("デモ数と成功率（更新回数はそろえてある）")
    axes[0].grid(alpha=0.3)

    for n_demos, sub in df.groupby("n_demos"):
        axes[1].scatter(sub["seed"], sub["success_rate"], label=f"デモ {int(n_demos)} 件")
    axes[1].set_xlabel("seed")
    axes[1].set_ylabel("eval_success_rate")
    axes[1].set_title("シードによるばらつき")
    axes[1].set_xticks(sorted(df["seed"].unique()))
    axes[1].grid(alpha=0.3)
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("データが足りません。ジョブの完了を待ってください。")

## 5. 考察

**ローカルでの実測結果と、そこから導かれる教訓は
[docs/05_BCを動かす.md](../docs/05_BCを動かす.md) の 5.4〜5.7 にまとめています。**

要点だけ先に挙げると:

- **更新回数をそろえても、デモ数によって成績が変わりました**
- しかも**単調ではありません**。デモが少なすぎても多すぎても落ちます
- 効いているのは **「データ量と反復回数のバランス」**
- **同一条件でもシードで大きく動く**ので、1 本の結果で判断してはいけません

## ✅ チェックリスト

- [ ] **3 条件の更新回数がそろっている**ことを 1. で確認した
- [ ] 9 本（3 条件 × 3 シード）を投入した
- [ ] **条件間の差とシード間のばらつきを比べて判定した**（4.）
- [ ] `bc_results.csv` を保存した
- [ ] `normalized_return` と `eval_success_rate` の両方を確認した

---

**次へ**: [docs/06_DAggerを試す.md](../docs/06_DAggerを試す.md) → [05_dagger_gail_job.ipynb](05_dagger_gail_job.ipynb)